# 🧠 Model Training & Comparison
## CreditLens AI — Loan Eligibility & Credit Decision System

**Objective:** Train, compare, and optimize machine learning models for loan approval prediction.

### Notebook Outline
1. Setup & Data Preparation
2. Model Training with Cross-Validation
3. Model Comparison & Selection
4. Hyperparameter Tuning (Optuna)
5. Final Evaluation on Test Set
6. Visualization of Results
7. Model Persistence & Summary

---
## 1. Setup & Data Preparation

In [ ]:
import sys, os, json, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (12, 6), 'font.size': 12, 'figure.dpi': 100})

FIGURES_DIR = Path('../reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {
    'primary': '#2563EB', 'secondary': '#7C3AED',
    'success': '#10B981', 'danger': '#EF4444',
    'warning': '#F59E0B', 'info': '#3B82F6',
    'models': ['#2563EB', '#7C3AED', '#10B981', '#F59E0B']
}

print('✅ Libraries loaded')

In [ ]:
# Load data using our pipeline modules
from src.data.merger import load_and_merge_datasets
from src.data.loader import validate_dataset, split_data
from src.features.engineer import engineer_features
from src.data.preprocessor import preprocess_data

# Step 1: Load & merge datasets
df = load_and_merge_datasets()
df = validate_dataset(df)
print(f'\nMerged dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')

# Step 2: Feature engineering
df = engineer_features(df)

# Step 3: Train/test split
X_train, X_test, y_train, y_test = split_data(df)
print(f'Training set: {X_train.shape[0]:,} rows')
print(f'Test set:     {X_test.shape[0]:,} rows')

# Step 4: Preprocessing (scaling, encoding, SMOTE)
X_train_proc, X_test_proc, y_train_res, _, feature_names = preprocess_data(
    X_train, X_test, y_train, apply_smote=True
)
print(f'\nAfter SMOTE: {X_train_proc.shape[0]:,} training samples')
print(f'Features: {X_train_proc.shape[1]}')

---
## 2. Model Training with Cross-Validation

In [ ]:
from src.models.trainer import get_models, cross_validate_model, train_single_model
from src.config import CV_FOLDS, PRIMARY_METRIC

models = get_models()
print(f'Training {len(models)} models with {CV_FOLDS}-fold cross-validation')
print(f'Primary metric: {PRIMARY_METRIC}\n')

# Cross-validate and train each model
cv_results = []
trained_models = {}

for name, model in models.items():
    # Cross-validate
    cv_result = cross_validate_model(model, X_train_proc, y_train_res, name)
    cv_results.append(cv_result)
    
    # Train on full training set
    fitted = train_single_model(model, X_train_proc, y_train_res, name)
    trained_models[name] = fitted

# Sort by score
cv_results.sort(key=lambda x: x['mean_score'], reverse=True)

# Display results
cv_df = pd.DataFrame(cv_results)
cv_df = cv_df[['model_name', 'mean_score', 'std_score']]
cv_df.columns = ['Model', f'Mean {PRIMARY_METRIC.upper()}', 'Std Dev']
cv_df.style.highlight_max(subset=[f'Mean {PRIMARY_METRIC.upper()}'], color='lightgreen')

In [ ]:
# Visualize cross-validation scores
fig, ax = plt.subplots(figsize=(10, 6))

model_names = [r['model_name'] for r in cv_results]
mean_scores = [r['mean_score'] for r in cv_results]
std_scores = [r['std_score'] for r in cv_results]

bars = ax.barh(model_names, mean_scores, xerr=std_scores,
               color=COLORS['models'][:len(model_names)],
               edgecolor='white', capsize=5, height=0.5)

for bar, score in zip(bars, mean_scores):
    ax.text(score + 0.005, bar.get_y() + bar.get_height()/2,
            f'{score:.4f}', va='center', fontweight='bold', fontsize=12)

ax.set_xlabel(f'{PRIMARY_METRIC.upper()} Score')
ax.set_title(f'Cross-Validation Results ({CV_FOLDS}-Fold)', fontsize=14, fontweight='bold')
ax.set_xlim(0.8, 1.0)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Model Comparison & Selection

In [ ]:
from src.models.evaluator import evaluate_all_models, create_comparison_table

# Evaluate all models on test set
results = evaluate_all_models(trained_models, X_test_proc, y_test)
comparison_df = create_comparison_table(results)

print('=' * 70)
print('TEST SET EVALUATION (Before Tuning)')
print('=' * 70)
comparison_df.style.highlight_max(color='lightgreen').format('{:.4f}')

In [ ]:
# Radar chart comparing all models across metrics
from matplotlib.patches import FancyBboxPatch

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
n_metrics = len(metrics)
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for i, (model_name, row) in enumerate(comparison_df.iterrows()):
    values = [row[m] for m in metrics]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=model_name,
            color=COLORS['models'][i], markersize=6)
    ax.fill(angles, values, alpha=0.1, color=COLORS['models'][i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0.5, 1.0)
ax.set_title('Model Performance Radar Chart', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'model_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Select best model
from src.models.trainer import select_best_model

best_name, best_model = select_best_model(trained_models, cv_results)
print(f'\n🏆 Best model (before tuning): {best_name}')
print(f'   AUC-ROC: {comparison_df.loc[best_name, "AUC-ROC"]:.4f}')
print(f'   F1-Score: {comparison_df.loc[best_name, "F1-Score"]:.4f}')

---
## 4. Hyperparameter Tuning (Optuna)

In [ ]:
from src.models.tuner import tune_model

print(f'🔬 Tuning {best_name} with Optuna (50 trials)...')
print('This may take several minutes.\n')

tuned_model, best_params, best_cv_score = tune_model(
    best_name, X_train_proc, y_train_res, n_trials=50, timeout=300
)

print(f'\n✅ Tuning complete!')
print(f'   Best CV Score ({PRIMARY_METRIC}): {best_cv_score:.4f}')
print(f'\n📋 Best Parameters:')
for param, value in best_params.items():
    print(f'   {param}: {value}')

In [ ]:
# Compare before vs after tuning
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, log_loss
)

# Predictions: before tuning
y_pred_before = best_model.predict(X_test_proc)
y_proba_before = best_model.predict_proba(X_test_proc)[:, 1]

# Predictions: after tuning
y_pred_after = tuned_model.predict(X_test_proc)
y_proba_after = tuned_model.predict_proba(X_test_proc)[:, 1]

before_metrics = {
    'Accuracy': accuracy_score(y_test, y_pred_before),
    'Precision': precision_score(y_test, y_pred_before),
    'Recall': recall_score(y_test, y_pred_before),
    'F1-Score': f1_score(y_test, y_pred_before),
    'AUC-ROC': roc_auc_score(y_test, y_proba_before),
}

after_metrics = {
    'Accuracy': accuracy_score(y_test, y_pred_after),
    'Precision': precision_score(y_test, y_pred_after),
    'Recall': recall_score(y_test, y_pred_after),
    'F1-Score': f1_score(y_test, y_pred_after),
    'AUC-ROC': roc_auc_score(y_test, y_proba_after),
}

tuning_comparison = pd.DataFrame({
    'Before Tuning': before_metrics,
    'After Tuning': after_metrics,
    'Improvement': {k: after_metrics[k] - before_metrics[k] for k in before_metrics}
}).round(4)

print('=' * 60)
print(f'{best_name}: BEFORE vs AFTER Hyperparameter Tuning')
print('=' * 60)
tuning_comparison.style.applymap(
    lambda v: 'color: green; font-weight: bold' if isinstance(v, float) and v > 0 else '',
    subset=['Improvement']
)

In [ ]:
# Visualize before vs after tuning
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(before_metrics))
width = 0.35

bars1 = ax.bar(x - width/2, list(before_metrics.values()), width,
               label='Before Tuning', color=COLORS['warning'], edgecolor='white')
bars2 = ax.bar(x + width/2, list(after_metrics.values()), width,
               label='After Tuning', color=COLORS['success'], edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(list(before_metrics.keys()), fontsize=11)
ax.set_ylabel('Score')
ax.set_ylim(0.7, 1.0)
ax.set_title(f'{best_name}: Before vs After Optuna Tuning', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'tuning_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Final Evaluation on Test Set

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, precision_recall_curve

# Use tuned model for final evaluation
final_model = tuned_model
y_pred_final = y_pred_after
y_proba_final = y_proba_after

# Classification report
print('=' * 60)
print(f'FINAL MODEL: {best_name} (Tuned)')
print('=' * 60)
print(classification_report(y_test, y_pred_final, target_names=['Rejected', 'Approved']))

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
cm = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Rejected', 'Approved'],
            yticklabels=['Rejected', 'Approved'],
            annot_kws={'size': 16})
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('Actual', fontsize=12)
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')

# Normalized
cm_norm = confusion_matrix(y_test, y_pred_final, normalize='true')
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', ax=axes[1],
            xticklabels=['Rejected', 'Approved'],
            yticklabels=['Rejected', 'Approved'],
            annot_kws={'size': 16})
axes[1].set_xlabel('Predicted', fontsize=12)
axes[1].set_ylabel('Actual', fontsize=12)
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold')

plt.suptitle(f'{best_name} (Tuned) — Confusion Matrix', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Visualization of Results

In [ ]:
# ROC Curve for all models + tuned model
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC Curves
for i, (name, model) in enumerate(trained_models.items()):
    y_proba = model.predict_proba(X_test_proc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})',
                color=COLORS['models'][i], linewidth=2)

# Add tuned model
fpr_t, tpr_t, _ = roc_curve(y_test, y_proba_final)
auc_t = roc_auc_score(y_test, y_proba_final)
axes[0].plot(fpr_t, tpr_t, label=f'{best_name} Tuned (AUC={auc_t:.4f})',
            color='black', linewidth=3, linestyle='--')

axes[0].plot([0, 1], [0, 1], 'k:', alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves', fontweight='bold')
axes[0].legend(fontsize=9)

# Precision-Recall Curves
for i, (name, model) in enumerate(trained_models.items()):
    y_proba = model.predict_proba(X_test_proc)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    axes[1].plot(recall, precision, label=name,
                color=COLORS['models'][i], linewidth=2)

precision_t, recall_t, _ = precision_recall_curve(y_test, y_proba_final)
axes[1].plot(recall_t, precision_t, label=f'{best_name} Tuned',
            color='black', linewidth=3, linestyle='--')

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves', fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Model Performance Curves', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Model Persistence & Summary

In [ ]:
from src.models.trainer import save_model

# Save the tuned model
save_model(tuned_model)
print(f'\n💾 Saved tuned {best_name} model to models/best_model.joblib')

# Save tuning results
tuning_results = {
    'best_model': best_name,
    'best_params': {k: float(v) if isinstance(v, (np.floating, np.integer)) else v
                    for k, v in best_params.items()},
    'best_cv_score': float(best_cv_score),
    'test_metrics_before': {k: float(v) for k, v in before_metrics.items()},
    'test_metrics_after': {k: float(v) for k, v in after_metrics.items()},
    'improvement': {k: float(after_metrics[k] - before_metrics[k]) for k in before_metrics},
}

reports_dir = Path('../reports')
with open(reports_dir / 'tuning_results.json', 'w') as f:
    json.dump(tuning_results, f, indent=2)
print(f'📊 Saved tuning results to reports/tuning_results.json')

In [ ]:
print('=' * 70)
print('   PHASE 3 SUMMARY: MODEL TRAINING & TUNING')
print('=' * 70)
print(f'''
📊 DATASET:
  • Merged: {df.shape[0]:,} rows from 2 sources
  • Features: {X_train_proc.shape[1]} (after preprocessing)
  • SMOTE applied: {X_train_proc.shape[0]:,} balanced training samples

🤖 MODELS TRAINED:
  1. Logistic Regression
  2. Random Forest
  3. XGBoost
  4. LightGBM

🏆 BEST MODEL: {best_name}
  Before tuning: AUC-ROC = {before_metrics["AUC-ROC"]:.4f}, F1 = {before_metrics["F1-Score"]:.4f}
  After tuning:  AUC-ROC = {after_metrics["AUC-ROC"]:.4f}, F1 = {after_metrics["F1-Score"]:.4f}

✅ TARGETS MET:
  AUC-ROC ≥ 0.85: {'✅ YES' if after_metrics["AUC-ROC"] >= 0.85 else '❌ NO'} ({after_metrics["AUC-ROC"]:.4f})
  F1-Score ≥ 0.80: {'✅ YES' if after_metrics["F1-Score"] >= 0.80 else '❌ NO'} ({after_metrics["F1-Score"]:.4f})

💾 ARTIFACTS SAVED:
  • models/best_model.joblib
  • models/preprocessor.joblib
  • reports/tuning_results.json
  • reports/model_comparison.json
''')

print('✅ Phase 3 Complete — Ready for Phase 4 (Explainability)')